### Import Dependencies

In [1]:
import openai
import os
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client

### Download all data from Qdrant

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [3]:
all_points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False,
)

In [4]:
all_points[0][0].payload

{'description': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sitting on to allow 

In [5]:
all_points[0]

[Record(id=0, payload={'description': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it i

In [6]:
all_context = [{"id": data.payload["parent_asin"], "text": data.payload["description"]} for data in all_points[0]]

In [7]:
all_context

[{'id': 'B0BRJS644Z',
  'text': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sitt

### Render a prompt to generate synthetic Eval reference dataset


In [8]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
        },
    },
}


SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, refer to the chunks as products.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""

In [9]:
print(SYSTEM_PROMPT)


I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, 

In [10]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with id and text:
[
  {
    "id": "B0BRJS644Z",
    "text": "Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) \u3010Solve Your Cooling Problem\u3011\u00a0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. \u3010Speed\u00a0Control\u00a0Switch\u3011 The speed controller located on the cord allows you to adjust the fan\u2019s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. \u3010High Compatibility USB-Powered\u3011 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving 

In [11]:
response = openai.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

```json
[
  {
    "question": "Which product is best if I need to cool my router or modem?",
    "chunk_ids": ["B0BRJS644Z"],
    "answer_example": "The Marame 120mm 5V USB Powered Fan is designed for cooling routers, modems, TV boxes, and similar electronics.",
    "reasoning": "This product explicitly mentions router and modem cooling, so it directly answers the question."
  },
  {
    "question": "Do you have headphones for kids that can be used both wirelessly and with a cable?",
    "chunk_ids": ["B09KQP2H7N"],
    "answer_example": "Yes, the kids wireless headphones support both Bluetooth and a 3.5mm wired connection.",
    "reasoning": "The product description states it works as both wireless and wired headphones and is made for kids."
  },
  {
    "question": "Is there an iPhone audio cable for connecting to a car stereo?",
    "chunk_ids": ["B0CC4HBS85"],
    "answer_example": "Yes, the Veetone Lightning to 3.5mm AUX cable is compatible with iPhones and can play audio through 

In [46]:
json_output = json.loads(response.choices[0].message.content)


In [47]:
json_output

[{'question': 'Do you have a fan I can use to cool my router or modem?',
  'chunk_ids': ['B0BRJS644Z'],
  'answer_example': 'Yes, the Marame 120mm 5V USB powered fan is designed for cooling routers, modems, DVRs, TV boxes, and other electronics.',
  'reasoning': 'This product explicitly mentions cooling routers, modems, and other electronics, so it directly answers the question.'},
 {'question': 'Which headphones are good for kids and have a volume limit?',
  'chunk_ids': ['B0BBF2VC6X', 'B09KQP2H7N'],
  'answer_example': 'We have kids headphones with safe volume features, including one model limited to 85dB and another with volume control and kid-friendly design.',
  'reasoning': 'Both products are kid-focused headphones, and one specifically highlights safe 85dB volume limiting while the other includes volume control and child-friendly features.'},
 {'question': 'Do you sell an iPhone aux cable for my car stereo?',
  'chunk_ids': ['B0CC4HBS85'],
  'answer_example': 'Yes, the Veetone L

In [48]:
len(json_output)

34

In [49]:
single_chunk_counter = 0
multiple_chunk_counter = 0
impossible_counter = 0

for item in json_output:
    if len(item["chunk_ids"]) == 1:
        single_chunk_counter += 1
    elif len(item["chunk_ids"]) > 1:
        multiple_chunk_counter += 1
    else:
        impossible_counter += 1

In [50]:
print(f"Single chunk questions: {single_chunk_counter}")
print(f"Multiple chunk questions: {multiple_chunk_counter}")
print(f"Impossible questions: {impossible_counter}")

Single chunk questions: 19
Multiple chunk questions: 10
Impossible questions: 5


In [51]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B0C4NJPN4Q")
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

In [52]:
points[0].payload

{'description': 'LP&No.1 Record Player with External Speakers, 3 Speed Vintage Belt-Drive Vinyl Turntable with Bluetooth Playback & Auto-Stop （Light Blue） 【3-Speed Vinyl Record player】 The diamond-tipped stylus plays all of 33 1/3, 45, and 78 RPMs records. Auto-stop setting makes it possible to stop automatically when it reaches the end of the record 【Stereo Turntable with Outstanding Sound】Equipped with a separable pair of speakers, let you enjoy the full,real and live vinyl audio in dual speakers easily.The volume is loud enough for many occasions 【Quality Vinyl Player and Speaker Set】Dual Bookshelf speakers allows for multiple placement options. You can make the speakers as a powerful sound base, or place it as a bookshelf or record shelf 【Modern Turntable with Bluetooth】Great record player as well as ability to work as a Bluetooth speaker.Turn to BT on your phone and hook up to any Bluetooth device easily for premium sound 【Worry-Free Warranty】 Lifetime technology-support with 30-d

In [53]:
def get_description(parent_asin: str) -> str:

    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        ),
        limit=100,
        with_payload=True,
        with_vectors=False
    )[0]

    return points[0].payload["description"]


In [54]:
get_description("B0C4NJPN4Q")

'LP&No.1 Record Player with External Speakers, 3 Speed Vintage Belt-Drive Vinyl Turntable with Bluetooth Playback & Auto-Stop （Light Blue） 【3-Speed Vinyl Record player】 The diamond-tipped stylus plays all of 33 1/3, 45, and 78 RPMs records. Auto-stop setting makes it possible to stop automatically when it reaches the end of the record 【Stereo Turntable with Outstanding Sound】Equipped with a separable pair of speakers, let you enjoy the full,real and live vinyl audio in dual speakers easily.The volume is loud enough for many occasions 【Quality Vinyl Player and Speaker Set】Dual Bookshelf speakers allows for multiple placement options. You can make the speakers as a powerful sound base, or place it as a bookshelf or record shelf 【Modern Turntable with Bluetooth】Great record player as well as ability to work as a Bluetooth speaker.Turn to BT on your phone and hook up to any Bluetooth device easily for premium sound 【Worry-Free Warranty】 Lifetime technology-support with 30-day money-back gu

### Create Eval dataset in LangSmith

In [55]:
ls_client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))

In [56]:
dataset_name = "rag-evaluation-dataset"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset"
)

In [57]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"],
        },
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(chunk_id) for chunk_id in item["chunk_ids"]]
        }
    )